In [ ]:
# Load processed + set parameter

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

DATA_PROCESSED_DIR = "../data/processed"

processed_feat_path = os.path.join(
    DATA_PROCESSED_DIR,
    "crypto_1d_2015-01-01_FEATURES_2017-11-09_to_2025-12-19.csv"  # sesuaikan namanya
)

df = pd.read_csv(processed_feat_path)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).sort_values(["Ticker","Date"]).reset_index(drop=True)

tickers = ["BTC-USD", "ETH-USD", "BNB-USD"]

LOOKBACK = 60
HORIZONS = [1, 7, 30]

FEATURE_COLS = ["Open", "High", "Low", "Close", "Volume"]  # OHLCV
TARGET_COL = "Close"

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9
df.head()

In [ ]:
# Transform Volume biar tidak “meledak”
# Volume sering skewed banget. Ini bikin training lebih stabil.

df = df.copy()
df["Volume"] = np.log1p(df["Volume"])  # log(1+Volume)

In [ ]:
# Fungsi windowing (X,y + meta tanggal)

def make_supervised_windows(dft: pd.DataFrame,
                            lookback: int,
                            horizon: int,
                            feature_cols: list[str],
                            target_col: str):
    dft = dft.sort_values("Date").reset_index(drop=True)

    dates = dft["Date"].to_numpy()
    Xraw = dft[feature_cols].to_numpy(dtype=float)
    yraw = dft[target_col].to_numpy(dtype=float)

    X_list, y_list = [], []
    target_dates, last_input_dates = [], []

    n = len(dft)
    for i in range(lookback - 1, n - horizon):
        x_win = Xraw[i - lookback + 1 : i + 1]   # (lookback, n_features)
        y_val = yraw[i + horizon]                # scalar

        if np.isnan(x_win).any() or np.isnan(y_val):
            continue

        X_list.append(x_win)
        y_list.append(y_val)
        last_input_dates.append(dates[i])
        target_dates.append(dates[i + horizon])

    X = np.array(X_list, dtype=float)
    y = np.array(y_list, dtype=float)

    meta = pd.DataFrame({
        "last_input_date": last_input_dates,
        "target_date": target_dates
    })
    return X, y, meta

In [ ]:
# Split time-series (berdasarkan target_date)

def split_by_time(X, y, meta, train_ratio=0.7, val_ratio=0.15):
    meta = meta.reset_index(drop=True)
    n = len(meta)

    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))

    idx_train = np.arange(0, train_end)
    idx_val   = np.arange(train_end, val_end)
    idx_test  = np.arange(val_end, n)

    return (
        X[idx_train], y[idx_train], meta.loc[idx_train].reset_index(drop=True),
        X[idx_val],   y[idx_val],   meta.loc[idx_val].reset_index(drop=True),
        X[idx_test],  y[idx_test],  meta.loc[idx_test].reset_index(drop=True)
    )

In [ ]:
# Scaling X (fit train saja) + simpan .npz

OUT_DIR = os.path.join(DATA_PROCESSED_DIR, f"datasets_close_OHLCV_L{LOOKBACK}")
os.makedirs(OUT_DIR, exist_ok=True)

def scale_3d_with_train_scaler(X_train, X_val, X_test):
    """
    Fit scaler pada train saja (semua fitur), lalu transform val/test.
    X shape: (n_samples, lookback, n_features)
    """
    scaler = StandardScaler()
    n_features = X_train.shape[-1]

    Xtr2 = X_train.reshape(-1, n_features)
    Xva2 = X_val.reshape(-1, n_features)
    Xte2 = X_test.reshape(-1, n_features)

    scaler.fit(Xtr2)

    X_train_s = scaler.transform(Xtr2).reshape(X_train.shape)
    X_val_s   = scaler.transform(Xva2).reshape(X_val.shape)
    X_test_s  = scaler.transform(Xte2).reshape(X_test.shape)

    return X_train_s, X_val_s, X_test_s, scaler


for t in tickers:
    dft = df[df["Ticker"] == t].copy()
    dft = dft.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)

    for H in HORIZONS:
        X, y, meta = make_supervised_windows(
            dft=dft,
            lookback=LOOKBACK,
            horizon=H,
            feature_cols=FEATURE_COLS,
            target_col=TARGET_COL
        )

        (X_tr, y_tr, m_tr,
        X_va, y_va, m_va,
        X_te, y_te, m_te) = split_by_time(
            X, y, meta, TRAIN_RATIO, VAL_RATIO
        )

        X_tr_s, X_va_s, X_te_s, scaler = scale_3d_with_train_scaler(X_tr, X_va, X_te)

        out_npz = os.path.join(OUT_DIR, f"{t}_H{H}_targetClose_OHLCV.npz")
        np.savez_compressed(
            out_npz,
            X_train=X_tr_s, y_train=y_tr,
            X_val=X_va_s,   y_val=y_va,
            X_test=X_te_s,  y_test=y_te,
            train_target_date=m_tr["target_date"].astype(str).to_numpy(),
            val_target_date=m_va["target_date"].astype(str).to_numpy(),
            test_target_date=m_te["target_date"].astype(str).to_numpy(),
            feature_cols=np.array(FEATURE_COLS, dtype=object),
            target_col=np.array([TARGET_COL], dtype=object),
            lookback=np.array([LOOKBACK]),
            horizon=np.array([H]),
            scaler_mean=scaler.mean_,
            scaler_scale=scaler.scale_
        )

        print(f"[SAVED] {out_npz}")
        print(f"  X_train: {X_tr_s.shape} | X_val: {X_va_s.shape} | X_test: {X_te_s.shape}")
        print(f"  target test range: {m_te['target_date'].min().date()} -> {m_te['target_date'].max().date()}")
        print("-" * 70)

In [ ]:
# Quick load test (pastikan fitur jadi 5)

sample_npz = os.path.join(OUT_DIR, "BTC-USD_H1_targetClose_OHLCV.npz")
d = np.load(sample_npz, allow_pickle=True)

print("X_train:", d["X_train"].shape)  # harusnya (..., 60, 5)
print("y_train:", d["y_train"].shape)
print("feature_cols:", d["feature_cols"])
print("lookback:", d["lookback"], "horizon:", d["horizon"])
print("test target range:", d["test_target_date"][0], "->", d["test_target_date"][-1])
